In [0]:
print("Databricks pipeline triggered successfully!")

Databricks pipeline triggered successfully!


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

source_path = "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/"
processed_path = "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/processed_user_data/"

print("Starting user data processing...")

# Read Parquet file copied by ADF
userDF = (
    spark.read
    .format("parquet")
    .load(source_path)
)

print("Source data:")
display(userDF)

print("Number of records:", userDF.count())

Starting user data processing...
Source data:


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6863156963309928>, line 17
     10 userDF = (
     11     spark.read
     12     .format("parquet")
     13     .load(source_path)
     14 )
     16 print("Source data:")
---> 17 display(userDF)
     19 print("Number of records:", userDF.count())

File <command-6863156963309928>, line 11
      7 print("Starting user data processing...")
      9 # Read Parquet file copied by ADF
     10 userDF = (
---> 11     spark.read
     12     .format("parquet")
     13     .load(source_path)
     14 )
     16 print("Source data:")
     17 display(userDF)

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for Parquet. It must be specified manually. SQLSTATE: 42KD9

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.dataSchemaNotSpecifiedError(QueryCompilationE

In [0]:
display(
    dbutils.fs.ls(
        "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/"
    )
)

path,name,size,modificationTime
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/chunk1.parquet,chunk1.parquet,731432,1789351889000
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/chunk2.parquet,chunk2.parquet,728056,1789351889000
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/chunk3.parquet,chunk3.parquet,733179,1789351889000
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/to_processed_user_data/chunk4.parquet,chunk4.parquet,729607,1789351889000


In [0]:
# User data transformations

userDF = userDF.withColumn(
    "hasanyapp",
    col("hasAnyApp").cast("boolean")
)

userDF = userDF.withColumn(
    "hasandroidapp",
    col("hasAndroidApp").cast("boolean")
)

userDF = userDF.withColumn(
    "hasiosapp",
    col("hasIosApp").cast("boolean")
)

userDF = userDF.withColumn(
    "hasprofilepicture",
    col("hasProfilePicture").cast("boolean")
)

userDF = userDF.withColumn(
    "socialNbFollowers",
    col("socialNbFollowers").cast(IntegerType())
)

userDF = userDF.withColumn(
    "socialNbFollows",
    col("socialNbFollows").cast(IntegerType())
)

userDF = userDF.withColumn(
    "productsPassRate",
    col("productsPassRate").cast(DecimalType(10, 2))
)

userDF = userDF.withColumn(
    "seniorityAsMonths",
    col("seniorityAsMonths").cast(DecimalType(10, 2))
)

userDF = userDF.withColumn(
    "seniorityAsYears",
    col("seniorityAsYears").cast(DecimalType(10, 2))
)

userDF = userDF.withColumn(
    "daysSinceLastLogin",
    when(
        col("daysSinceLastLogin").isNotNull(),
        col("daysSinceLastLogin").cast(IntegerType())
    ).otherwise(0)
)

display(userDF)

In [0]:
silver_path = "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/silver/users/"

userDF.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_path)

print("User data successfully written to Silver!")

User data successfully written to Silver!


In [0]:
display(
    spark.read
    .format("delta")
    .load(silver_path)
)

In [0]:
# Move processed files

files = dbutils.fs.ls(source_path)

for file in files:
    dbutils.fs.mv(
        file.path,
        processed_path + file.name
    )

print("Files successfully moved to processed_user_data!")

Files successfully moved to processed_user_data!


In [0]:
display(
    dbutils.fs.ls(
        "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/processed_user_data/"
    )
)

path,name,size,modificationTime
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/processed_user_data/chunk1.parquet,chunk1.parquet,731432,1789283988000
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/processed_user_data/chunk2.parquet,chunk2.parquet,728056,1789283989000
abfss://landing-zone-2@ecomadls.dfs.core.windows.net/processed_user_data/chunk3.parquet,chunk3.parquet,733179,1789283990000


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Parameter received from ADF
dbutils.widgets.text("input_file", "")

input_file = dbutils.widgets.get("input_file")

if not input_file:
    raise ValueError("input_file parameter was not provided by ADF")

source_path = (
    "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/"
    "to_processed_user_data/"
    + input_file
)

processed_path = (
    "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/"
    "processed_user_data/"
    + input_file
)

silver_path = (
    "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/"
    "silver/users/"
)

print("Input file:", input_file)
print("Source:", source_path)

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-7077718584341951>, line 11
      8 input_file = dbutils.widgets.get("input_file")
     10 if not input_file:
---> 11     raise ValueError("input_file parameter was not provided by ADF")
     13 source_path = (
     14     "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/"
     15     "to_processed_user_data/"
     16     + input_file
     17 )
     19 processed_path = (
     20     "abfss://landing-zone-2@ecomadls.dfs.core.windows.net/"
     21     "processed_user_data/"
     22     + input_file
     23 )

ValueError: input_file parameter was not provided by ADF